# 11. Wybór i łączenie falek w praktyce

**Arkadiusz Liber** · *Falki w praktyce. Od wzoru do kodu*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/writingin/falki-w-praktyce-od-wzoru-do-kodu-notatniki/blob/main/rozdzial-11.ipynb)

Notatnik do książki. Zawiera wszystkie programy z tekstu rozdziału oraz ćwiczenia z rozwiązaniami.

Notatnik działa lokalnie (Jupyter) i w Google Colab. Komórki można uruchamiać po kolei poleceniem *Uruchom wszystko*.

In [1]:
# Uruchom tę komórkę jako pierwszą (lokalnie albo w Google Colab).
import sys, subprocess, importlib.util, os
if importlib.util.find_spec("pywt") is None:      # PyWavelets
    subprocess.check_call([sys.executable, "-m", "pip",
                           "install", "-q", "PyWavelets"])
import numpy as np
import matplotlib.pyplot as plt
import pywt

SZARY, NIEBIESKI, RDZAWY, ZIELONY = "#757575", "#1273E2", "#D93B0B", "#078832"
FIOLET, BURSZTYN = "#9258D5", "#B36000"
plt.rcParams.update({
    "text.color": SZARY, "axes.labelcolor": SZARY, "axes.edgecolor": SZARY,
    "xtick.color": SZARY, "ytick.color": SZARY, "axes.titlecolor": SZARY,
    "font.size": 10, "axes.titlesize": 10,
    "axes.spines.top": False, "axes.spines.right": False})

# Dane z obserwatorium SDO używane w tym rozdziale.
# Courtesy of NASA/SDO and the AIA, EVE, and HMI science teams
DANE = "dane/sdo-20240115.npz"
if not os.path.exists(DANE):                 # np. Google Colab: pobranie z repozytorium
    import urllib.request
    os.makedirs("dane", exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/writingin/falki-w-praktyce-od-wzoru-do-kodu-notatniki/main/dane/sdo-20240115.npz", DANE)


## Programy z tekstu rozdziału

### Kod 11.1. Odszumianie czterech sygnałów ośmioma falkami: średni stosunek sygnału do szumu z 20 prób

In [2]:
import numpy as np
import pywt

N = 1024
x = np.arange(N)/N

def sygnaly_testowe():
    schodkowy = np.zeros(N)
    for p, h in zip([0.1, 0.13, 0.25, 0.4, 0.44, 0.65, 0.76, 0.81],
                    [4, -5, 3, -4, 5, -4.2, 2.1, 4.3]):
        schodkowy += h*(x > p)
    szczyty = sum(h*(1 + np.abs((x - p)/w))**-4 for p, h, w in zip(
        [0.1, 0.2, 0.35, 0.5, 0.62, 0.78, 0.9], [4, 5, 3, 4, 5, 4.2, 3],
        [0.005, 0.008, 0.006, 0.01, 0.01, 0.03, 0.005]))
    gladki = 4*np.sin(4*np.pi*x) - np.sign(x - 0.3) - np.sign(0.72 - x)
    oscylacyjny = np.sqrt(x*(1 - x))*np.sin(2.1*np.pi/(x + 0.05))
    s = {"schodkowy": schodkowy, "szczyty": szczyty,
         "gładki ze skokami": gladki, "oscylacyjny": oscylacyjny}
    return {k: 7*v/np.std(v) for k, v in s.items()}

def odszum(z, falka, poziomy=5):
    wsp = pywt.wavedec(z, falka, level=poziomy, mode="periodization")
    sigma = np.median(np.abs(wsp[-1]))/0.6745
    prog = sigma*np.sqrt(2*np.log(len(z)))
    wsp = [wsp[0]] + [pywt.threshold(d, prog, "hard") for d in wsp[1:]]
    return pywt.waverec(wsp, falka, mode="periodization")

snr = lambda s, y: 10*np.log10(np.sum(s**2)/np.sum((s - y)**2))
falki = ["haar", "db2", "db4", "db8", "sym4", "sym8", "coif3", "bior4.4"]
rng = np.random.default_rng(0)

print(" "*18 + "".join(f"{f:>8s}" for f in falki))
for nazwa, s in sygnaly_testowe().items():
    wynik = np.zeros(len(falki))
    for proba in range(20):
        z = s + rng.normal(size=N)
        wynik += [snr(s, odszum(z, f)) for f in falki]
    print(f"{nazwa:18s}" + "".join(f"{w:8.2f}" for w in wynik/20))

                      haar     db2     db4     db8    sym4    sym8   coif3 bior4.4
schodkowy            28.29   24.88   22.83   20.43   23.50   22.31   22.25   23.66
szczyty              20.45   21.64   22.38   20.17   21.43   20.75   20.92   21.46
gładki ze skokami    22.23   27.30   26.71   26.78   28.23   27.40   26.96   27.42
oscylacyjny          17.03   21.05   22.98   23.18   22.92   24.22   23.76   23.01


### Kod 11.2. Kompresja trzech obrazów do 5% współczynników: porównanie falek

In [3]:
import numpy as np
import pywt

def scena(n=256):
    y, x = np.mgrid[0:n, 0:n]/n
    F = 60 + 80*x + 40*np.sin(2*np.pi*y)*x
    F += 90*((x - 0.3)**2 + (y - 0.35)**2 < 0.15**2)
    F -= 50*((np.abs(x - 0.7) < 0.15) & (np.abs(y - 0.65) < 0.2))
    pas = y - 0.15*(x - 0.5)
    F += 70*((pas > 0.75) & (pas < 0.8))
    return np.clip(F, 0, 255)

def kompresja(F, falka, odsetek=0.05, poziomy=4):
    wsp = pywt.wavedec2(F, falka, level=poziomy, mode="periodization")
    tab, uklad = pywt.coeffs_to_array(wsp)
    prog = np.sort(np.abs(tab).ravel())[-int(odsetek*tab.size)]
    tab = np.where(np.abs(tab) >= prog, tab, 0)
    G = pywt.waverec2(pywt.array_to_coeffs(
        tab, uklad, output_format="wavedec2"), falka,
        mode="periodization")
    return 10*np.log10(255**2/np.mean((F - G)**2))

dane = np.load("dane/sdo-20240115.npz")
obrazy = {"pętle": 255*np.sqrt(dane["aia171"]/dane["aia171"].max()),
          "plama": 255*dane["hmi_ic_plama"]/dane["hmi_ic_plama"].max(),
          "scena": scena()}
falki = ["haar", "db2", "sym4", "sym8", "coif2", "bior2.2", "bior4.4"]
print(" "*11 + "".join(f"{f:>9s}" for f in falki))
for nazwa, F in obrazy.items():
    print(f"{nazwa:11s}" + "".join(f"{kompresja(F, f):9.2f}" for f in falki))

                haar      db2     sym4     sym8    coif2  bior2.2  bior4.4
pętle          39.07    40.96    41.67    41.89    41.74    41.51    41.96
plama          32.84    33.44    33.80    33.98    33.87    33.37    33.92
scena          52.45    50.25    46.04    43.61    46.35    55.19    47.44


### Kod 11.3. Skok w próbce 517: liczba dużych współczynników i położenie największego

In [4]:
import numpy as np
import pywt

N = 1024
x = np.arange(N)/N
s = np.sin(2*np.pi*2*x) + 1.0*(np.arange(N) >= 517)

print("falka    filtr   poziom 1          poziom 3")
for falka in ("haar", "db2", "db4", "db8", "sym8", "coif3", "bior4.4"):
    opis = f"{falka:8s} {pywt.Wavelet(falka).dec_len:4d}  "
    wsp = pywt.wavedec(s, falka, level=3, mode="periodization")
    for poziom, cD, brzeg in ((1, wsp[3], 16), (3, wsp[1], 4)):
        c = np.abs(cD[brzeg:-brzeg])           # bez brzegów
        duze = int(np.sum(c > 0.05*c.max()))
        polozenie = 2**poziom*(int(np.argmax(c)) + brzeg)
        opis += f"  duże: {duze:2d}, próbka {polozenie:3d}"
    print(opis)

falka    filtr   poziom 1          poziom 3
haar        2    duże:  1, próbka 516  duże: 40, próbka 512
db2         4    duże:  1, próbka 516  duże:  2, próbka 512
db4         8    duże:  3, próbka 514  duże:  3, próbka 512
db8        16    duże:  4, próbka 510  duże:  6, próbka 504
sym8       16    duże:  4, próbka 516  duże:  5, próbka 512
coif3      18    duże:  3, próbka 514  duże:  4, próbka 512
bior4.4    10    duże:  3, próbka 516  duże:  3, próbka 512


### Kod 11.4. Błąd odtworzenia sygnału falką inną niż użyta do rozkładu

In [5]:
import numpy as np
import pywt

x = np.arange(1024)/1024
s = 4*np.sin(4*np.pi*x) - np.sign(x - 0.3) - np.sign(0.72 - x)

for analiza, syntezy in (("db4", ("db4", "sym4", "db2", "haar")),
                         ("bior2.2", ("bior2.2", "rbio2.2", "bior4.4"))):
    wsp = pywt.wavedec(s, analiza, level=4, mode="periodization")
    for synteza in syntezy:
        y = pywt.waverec(wsp, synteza, mode="periodization")
        blad = 100*np.linalg.norm(y - s)/np.linalg.norm(s)
        print(f"analiza {analiza:8s} synteza {synteza:8s} błąd {blad:6.2f}%")

analiza db4      synteza db4      błąd   0.00%
analiza db4      synteza sym4     błąd  53.34%
analiza db4      synteza db2      błąd  30.72%
analiza db4      synteza haar     błąd  45.33%
analiza bior2.2  synteza bior2.2  błąd   0.00%
analiza bior2.2  synteza rbio2.2  błąd   6.20%
analiza bior2.2  synteza bior4.4  błąd   2.20%


### Kod 11.5. Transformata z inną falką na każdym poziomie

In [6]:
import numpy as np
import pywt

def analiza(s, falki):
    detale, a = [], s
    for f in falki:                       # od poziomu najdrobniejszego
        a, d = pywt.dwt(a, f, mode="periodization")
        detale.append(d)
    return a, detale

def synteza(a, detale, falki):
    for f, d in zip(reversed(falki), reversed(detale)):
        a = pywt.idwt(a, d, f, mode="periodization")
    return a

x = np.arange(1024)/1024
s = 4*np.sin(4*np.pi*x) - np.sign(x - 0.3) - np.sign(0.72 - x)
falki = ["haar", "db4", "sym8"]

a, detale = analiza(s, falki)
energia = np.sum(a**2) + sum(np.sum(d**2) for d in detale)
print(np.allclose(synteza(a, detale, falki), s),
      np.isclose(energia, np.sum(s**2)))
zla = synteza(a, detale, falki[::-1])     # falki w złej kolejności
print(f"{100*np.linalg.norm(zla - s)/np.linalg.norm(s):.2f}%")

True True
4.38%


### Kod 11.6. Obraz schodkowy w jednym kierunku i gładki w drugim: jedna falka i dwie falki

In [7]:
import numpy as np
import pywt

n = 256
y, x = np.mgrid[0:n, 0:n]/n
rng = np.random.default_rng(3)
warstwy = np.repeat(rng.uniform(40, 200, size=8), n//8)   # skoki wzdłuż y
F = warstwy[:, None]*(0.6 + 0.4*np.sin(3*np.pi*x + 1))    # gładko wzdłuż x

def psnr_po_kompresji(falka, odsetek):
    wsp = pywt.wavedec2(F, falka, level=4, mode="periodization")
    tab, uklad = pywt.coeffs_to_array(wsp)
    prog = np.sort(np.abs(tab).ravel())[-int(odsetek*tab.size)]
    G = pywt.waverec2(pywt.array_to_coeffs(
        np.where(np.abs(tab) >= prog, tab, 0), uklad,
        output_format="wavedec2"), falka, mode="periodization")
    return 10*np.log10(255**2/np.mean((F - G)**2))

for falka in ("haar", "sym8", ("haar", "sym8"), ("sym8", "haar")):
    wyniki = [psnr_po_kompresji(falka, o) for o in (0.005, 0.01, 0.02)]
    print(f"{str(falka):18s}" + "".join(f"{w:8.2f}" for w in wyniki))

haar                 37.53   42.53   46.46
sym8                 29.20   32.95   37.39
('haar', 'sym8')     37.87   49.16   66.73
('sym8', 'haar')     29.41   32.76   36.43


### Kod 11.7. Słownik złożony z bazy Haara i bazy kosinusowej: spójność i pogoń za dopasowaniem

In [8]:
import numpy as np
import pywt
from scipy.fft import dct, idct

N = 1024
x = np.arange(N)/N
schodki = np.zeros(N)
for p, h in zip([0.15, 0.4, 0.55, 0.8], [2, -3, 2.5, -1.5]):
    schodki += h*(x > p)
jedn = np.eye(N)
oscylacje = (30*idct(jedn[75], norm="ortho")
             + 20*idct(jedn[181], norm="ortho"))
s = schodki + oscylacje

uklad = pywt.coeffs_to_array(
    pywt.wavedec(s, "haar", mode="periodization"))[1]
haar_a = lambda v: pywt.coeffs_to_array(
    pywt.wavedec(v, "haar", mode="periodization"))[0]
haar_s = lambda c: pywt.waverec(pywt.array_to_coeffs(
    c, uklad, output_format="wavedec"), "haar", mode="periodization")
kos_a = lambda v: dct(v, norm="ortho")
kos_s = lambda c: idct(c, norm="ortho")

# spójność wzajemna baz: iloczyny skalarne wszystkich par atomów
H = np.array([haar_s(e) for e in jedn])
print("spójność Haar–kosinusy:",
      round(float(np.abs(dct(H, norm="ortho", axis=1)).max()), 3),
      " impulsy–kosinusy:",
      round(float(np.abs(dct(jedn, norm="ortho", axis=1)).max()), 3))

def ile_atomow(c, blad=0.01):
    e = np.sort(c**2)[::-1]
    reszta = np.sqrt(np.clip(e.sum() - np.cumsum(e), 0, None)/e.sum())
    return int(np.argmax(reszta <= blad)) + 1

print("baza Haara:", ile_atomow(haar_a(s)),
      " baza kosinusowa:", ile_atomow(kos_a(s)))

reszta, a, b, kroki = s.copy(), np.zeros(N), np.zeros(N), 0
while np.linalg.norm(reszta) > 0.01*np.linalg.norm(s):
    ch, ck = haar_a(reszta), kos_a(reszta)
    i, j = np.argmax(np.abs(ch)), np.argmax(np.abs(ck))
    atom = np.zeros(N)
    if abs(ch[i]) >= abs(ck[j]):               # atom z bazy Haara
        a[i] += ch[i]; atom[i] = ch[i]; reszta = reszta - haar_s(atom)
    else:                                      # atom z bazy kosinusowej
        b[j] += ck[j]; atom[j] = ck[j]; reszta = reszta - kos_s(atom)
    kroki += 1
print("słownik, pogoń za dopasowaniem:", kroki, "kroków,",
      int(np.sum(a != 0)), "atomów Haara,",
      int(np.sum(b != 0)), "atomów kosinusowych")
blad = lambda p, w: np.linalg.norm(p - w)/np.linalg.norm(w)
print("błąd części schodkowej:", round(float(blad(haar_s(a), schodki)), 2),
      " oscylacyjnej:", round(float(blad(kos_s(b), oscylacje)), 2))

spójność Haar–kosinusy: 1.0  impulsy–kosinusy: 0.044
baza Haara: 914  baza kosinusowa: 546
słownik, pogoń za dopasowaniem: 198 kroków, 108 atomów Haara, 71 atomów kosinusowych
błąd części schodkowej: 0.78  oscylacyjnej: 0.9


### Kod 11.8. Naprzemienne progowanie w bazie Haara i w bazie kosinusowej

In [9]:
import numpy as np
import pywt
from scipy.fft import dct, idct

N = 1024
x = np.arange(N)/N
schodki = np.zeros(N)
for p, h in zip([0.15, 0.4, 0.55, 0.8], [2, -3, 2.5, -1.5]):
    schodki += h*(x > p)
jedn = np.eye(N)
s = schodki + 30*idct(jedn[75], norm="ortho") \
            + 20*idct(jedn[181], norm="ortho")

uklad = pywt.coeffs_to_array(
    pywt.wavedec(s, "haar", mode="periodization"))[1]
haar_a = lambda v: pywt.coeffs_to_array(
    pywt.wavedec(v, "haar", mode="periodization"))[0]
haar_s = lambda c: pywt.waverec(pywt.array_to_coeffs(
    c, uklad, output_format="wavedec"), "haar", mode="periodization")

def dwie_bazy(z, prog_koncowy, kroki=60):
    u, v = np.zeros(N), np.zeros(N)
    start = max(np.abs(haar_a(z)).max(), np.abs(dct(z, norm="ortho")).max())
    for prog in np.geomspace(start, prog_koncowy, kroki):
        u = haar_s(pywt.threshold(haar_a(z - v), prog, "hard"))
        v = idct(pywt.threshold(dct(z - u, norm="ortho"), prog, "hard"),
                 norm="ortho")
    return u + v

def jedna_baza(z, falka, sigma):
    wsp = pywt.wavedec(z, falka, level=5, mode="periodization")
    prog = sigma*np.sqrt(2*np.log(N))
    wsp = [wsp[0]] + [pywt.threshold(d, prog, "hard") for d in wsp[1:]]
    return pywt.waverec(wsp, falka, mode="periodization")

snr = lambda y: 10*np.log10(np.sum(s**2)/np.sum((s - y)**2))
sigma, wyniki = 0.4, np.zeros(5)
rng = np.random.default_rng(1)
for proba in range(10):
    z = s + sigma*rng.normal(size=N)
    kos = idct(pywt.threshold(dct(z, norm="ortho"),
               sigma*np.sqrt(2*np.log(N)), "hard"), norm="ortho")
    wyniki += [snr(z), snr(jedna_baza(z, "haar", sigma)),
               snr(jedna_baza(z, "sym8", sigma)), snr(kos),
               snr(dwie_bazy(z, 3*sigma))]
for nazwa, w in zip(("zaszumiony", "Haar", "sym8", "kosinusy",
                     "dwie bazy"), wyniki/10):
    print(f"{nazwa:11s} {w:6.2f} dB")

zaszumiony   12.82 dB
Haar         10.40 dB
sym8         13.31 dB
kosinusy     17.45 dB
dwie bazy    19.93 dB


### Kod 11.9. Uśrednianie wyników odszumiania po falkach i po przesunięciach

In [10]:
import numpy as np
import pywt

N = 1024
x = np.arange(N)/N

def sygnaly_testowe():
    schodkowy = np.zeros(N)
    for p, h in zip([0.1, 0.13, 0.25, 0.4, 0.44, 0.65, 0.76, 0.81],
                    [4, -5, 3, -4, 5, -4.2, 2.1, 4.3]):
        schodkowy += h*(x > p)
    szczyty = sum(h*(1 + np.abs((x - p)/w))**-4 for p, h, w in zip(
        [0.1, 0.2, 0.35, 0.5, 0.62, 0.78, 0.9], [4, 5, 3, 4, 5, 4.2, 3],
        [0.005, 0.008, 0.006, 0.01, 0.01, 0.03, 0.005]))
    gladki = 4*np.sin(4*np.pi*x) - np.sign(x - 0.3) - np.sign(0.72 - x)
    oscylacyjny = np.sqrt(x*(1 - x))*np.sin(2.1*np.pi/(x + 0.05))
    s = {"schodkowy": schodkowy, "szczyty": szczyty,
         "gładki ze skokami": gladki, "oscylacyjny": oscylacyjny}
    return {k: 7*v/np.std(v) for k, v in s.items()}

def odszum(z, falka, poziomy=5):
    wsp = pywt.wavedec(z, falka, level=poziomy, mode="periodization")
    sigma = np.median(np.abs(wsp[-1]))/0.6745
    prog = sigma*np.sqrt(2*np.log(len(z)))
    wsp = [wsp[0]] + [pywt.threshold(d, prog, "hard") for d in wsp[1:]]
    return pywt.waverec(wsp, falka, mode="periodization")

def po_przesunieciach(z, falka, ile=16):
    return np.mean([np.roll(odszum(np.roll(z, k), falka), -k)
                    for k in range(ile)], axis=0)

snr = lambda s, y: 10*np.log10(np.sum(s**2)/np.sum((s - y)**2))
rng = np.random.default_rng(2)
kolumny = ("haar", "sym8", "średnia 3", "przes. haar", "przes. sym8",
           "wszystko")
print(" "*18 + "".join(f"{k:>12s}" for k in kolumny))
for nazwa, s in sygnaly_testowe().items():
    w = np.zeros(6)
    for proba in range(10):
        z = s + rng.normal(size=N)
        d = {f: odszum(z, f) for f in ("haar", "db4", "sym8")}
        ph, ps = po_przesunieciach(z, "haar"), po_przesunieciach(z, "sym8")
        w += [snr(s, d["haar"]), snr(s, d["sym8"]),
              snr(s, (d["haar"] + d["db4"] + d["sym8"])/3),
              snr(s, ph), snr(s, ps), snr(s, (ph + ps)/2)]
    print(f"{nazwa:18s}" + "".join(f"{v:12.2f}" for v in w/10))

                          haar        sym8   średnia 3 przes. haar przes. sym8    wszystko
schodkowy                28.48       22.52       26.68       31.55       25.74       29.01
szczyty                  20.49       20.84       24.06       24.79       24.04       25.50
gładki ze skokami        21.87       26.72       27.24       27.17       28.45       28.69
oscylacyjny              17.17       24.02       23.27       22.41       26.85       25.20


## Funkcje pomocnicze

Funkcje i dane, z których korzystają rozwiązania ćwiczeń.

In [11]:
N = 1024
x = np.arange(N)/N

def sygnaly_testowe():
    schodkowy = np.zeros(N)
    for p, h in zip([0.1, 0.13, 0.25, 0.4, 0.44, 0.65, 0.76, 0.81],
                    [4, -5, 3, -4, 5, -4.2, 2.1, 4.3]):
        schodkowy += h*(x > p)
    szczyty = sum(h*(1 + np.abs((x - p)/w))**-4 for p, h, w in zip(
        [0.1, 0.2, 0.35, 0.5, 0.62, 0.78, 0.9], [4, 5, 3, 4, 5, 4.2, 3],
        [0.005, 0.008, 0.006, 0.01, 0.01, 0.03, 0.005]))
    gladki = 4*np.sin(4*np.pi*x) - np.sign(x - 0.3) - np.sign(0.72 - x)
    oscylacyjny = np.sqrt(x*(1 - x))*np.sin(2.1*np.pi/(x + 0.05))
    s = {"schodkowy": schodkowy, "szczyty": szczyty,
         "gładki ze skokami": gladki, "oscylacyjny": oscylacyjny}
    return {k: 7*v/np.std(v) for k, v in s.items()}

def odszum(z, falka, poziomy=5):
    wsp = pywt.wavedec(z, falka, level=poziomy, mode="periodization")
    sigma = np.median(np.abs(wsp[-1]))/0.6745
    prog = sigma*np.sqrt(2*np.log(len(z)))
    wsp = [wsp[0]] + [pywt.threshold(d, prog, "hard") for d in wsp[1:]]
    return pywt.waverec(wsp, falka, mode="periodization")

snr = lambda s, y: 10*np.log10(np.sum(s**2)/np.sum((s - y)**2))

## Ćwiczenia i rozwiązania

### Funkcje wspólne dla rozwiązań

In [12]:
N = 1024
x = np.arange(N)/N

def sygnaly_testowe():
    schodkowy = np.zeros(N)
    for p, h in zip([0.1, 0.13, 0.25, 0.4, 0.44, 0.65, 0.76, 0.81],
                    [4, -5, 3, -4, 5, -4.2, 2.1, 4.3]):
        schodkowy += h*(x > p)
    szczyty = sum(h*(1 + np.abs((x - p)/w))**-4 for p, h, w in zip(
        [0.1, 0.2, 0.35, 0.5, 0.62, 0.78, 0.9], [4, 5, 3, 4, 5, 4.2, 3],
        [0.005, 0.008, 0.006, 0.01, 0.01, 0.03, 0.005]))
    gladki = 4*np.sin(4*np.pi*x) - np.sign(x - 0.3) - np.sign(0.72 - x)
    oscylacyjny = np.sqrt(x*(1 - x))*np.sin(2.1*np.pi/(x + 0.05))
    s = {"schodkowy": schodkowy, "szczyty": szczyty,
         "gładki ze skokami": gladki, "oscylacyjny": oscylacyjny}
    return {k: 7*v/np.std(v) for k, v in s.items()}

def odszum(z, falka, poziomy=5, tryb="hard"):
    wsp = pywt.wavedec(z, falka, level=poziomy, mode="periodization")
    sigma = np.median(np.abs(wsp[-1]))/0.6745
    prog = sigma*np.sqrt(2*np.log(len(z)))
    wsp = [wsp[0]] + [pywt.threshold(d, prog, tryb) for d in wsp[1:]]
    return pywt.waverec(wsp, falka, mode="periodization")

snr = lambda s, y: 10*np.log10(np.sum(s**2)/np.sum((s - y)**2))

### Ćwiczenie 11.1
Powtórz doświadczenie z kodu 11.1 dla progowania miękkiego. Czy kolejność falek się zmienia?

### Rozwiązanie 11.1

In [13]:
falki = ["haar", "db2", "db4", "db8", "sym4", "sym8", "coif3", "bior4.4"]
rng = np.random.default_rng(0)
print(" "*18 + "".join(f"{f:>8s}" for f in falki))
for nazwa, s in sygnaly_testowe().items():
    w = np.zeros(len(falki))
    for proba in range(20):
        z = s + rng.normal(size=N)
        w += [snr(s, odszum(z, f, tryb="soft")) for f in falki]
    print(f"{nazwa:18s}" + "".join(f"{v:8.2f}" for v in w/20))
# Wyniki są o 2–4 dB niższe niż dla progowania twardego (próg uniwersalny
# jest dla progowania miękkiego za wysoki), ale kolejność falek jest podobna.

                      haar     db2     db4     db8    sym4    sym8   coif3 bior4.4
schodkowy            21.63   19.41   18.53   17.33   18.82   18.26   18.33   18.36
szczyty              15.17   16.38   16.12   14.63   16.20   15.73   15.99   16.13
gładki ze skokami    20.16   25.98   25.76   25.80   26.14   25.89   25.65   25.71
oscylacyjny          14.30   17.26   18.86   19.30   19.44   20.17   19.64   19.22


### Ćwiczenie 11.2
Rozszerz kod 11.3 o poziomy 2 i 4. Jak przesunięcie położenia największego współczynnika zależy od poziomu dla falek db4 i db8?

### Rozwiązanie 11.2

In [14]:
s = np.sin(2*np.pi*2*x) + 1.0*(np.arange(N) >= 517)
for falka in ("db4", "db8", "sym8"):
    wsp = pywt.wavedec(s, falka, level=4, mode="periodization")
    opis = f"{falka:5s}"
    for poziom in (1, 2, 3, 4):
        cD = wsp[5 - poziom]; brzeg = max(2, 32//2**poziom)
        k = int(np.argmax(np.abs(cD[brzeg:-brzeg]))) + brzeg
        opis += f"  poziom {poziom}: {2**poziom*k - 517:+4d}"
    print(opis)
# Przesunięcie (w próbkach sygnału) w przybliżeniu podwaja się z każdym poziomem.

db4    poziom 1:   -3  poziom 2:   -1  poziom 3:   -5  poziom 4:   -5
db8    poziom 1:   -7  poziom 2:   -9  poziom 3:  -13  poziom 4:  -21
sym8   poziom 1:   -1  poziom 2:   -1  poziom 3:   -5  poziom 4:   -5


### Ćwiczenie 11.3
Sprawdź dla czterech sygnałów testowych odszumianie z falką Haara na dwóch najdrobniejszych poziomach i falką sym8 na trzech pozostałych oraz układ odwrotny. Porównaj wyniki z pojedynczymi falkami.

### Rozwiązanie 11.3

In [15]:
def odszum_mieszany(z, falki):
    detale, a = [], z
    for f in falki:
        a, d = pywt.dwt(a, f, mode="periodization"); detale.append(d)
    sigma = np.median(np.abs(detale[0]))/0.6745
    prog = sigma*np.sqrt(2*np.log(len(z)))
    for f, d in zip(reversed(falki), reversed(detale)):
        a = pywt.idwt(a, pywt.threshold(d, prog, "hard"), f, mode="periodization")
    return a
uklady = {"haar": ["haar"]*5, "sym8": ["sym8"]*5,
          "haar×2, sym8×3": ["haar"]*2 + ["sym8"]*3,
          "sym8×2, haar×3": ["sym8"]*2 + ["haar"]*3}
rng = np.random.default_rng(1)
print(" "*18 + "".join(f"{k:>16s}" for k in uklady))
for nazwa, s in sygnaly_testowe().items():
    w = [np.mean([snr(s, odszum_mieszany(s + rng.normal(size=N), u))
                  for proba in range(20)]) for u in uklady.values()]
    print(f"{nazwa:18s}" + "".join(f"{v:16.2f}" for v in w))

                              haar            sym8  haar×2, sym8×3  sym8×2, haar×3
schodkowy                    28.53           22.48           24.39           24.53
szczyty                      20.31           20.67           19.26           22.05
gładki ze skokami            21.99           27.37           27.70           22.19
oscylacyjny                  17.13           24.35           19.60           19.74


### Ćwiczenie 11.4
Rozłóż obraz pętli koronalnych z kodu 11.2 z falką bior4.4 i odtwórz go falkami bior4.4, rbio4.4 i sym4. Oblicz PSNR.

### Rozwiązanie 11.4

In [16]:
dane = np.load("dane/sdo-20240115.npz")["aia171"]   # Courtesy of NASA/SDO and the AIA, EVE, and HMI science teams
F = 255*np.sqrt(dane/dane.max())
wsp = pywt.wavedec2(F, "bior4.4", level=3, mode="periodization")
for falka in ("bior4.4", "rbio4.4", "sym4"):
    G = pywt.waverec2(wsp, falka, mode="periodization")
    mse = np.mean((F - G)**2)
    print(falka, "PSNR:", "nieskończony" if mse < 1e-18 else f"{10*np.log10(255**2/mse):.2f} dB")
# bior4.4 jest prawie ortogonalna, więc synteza „odwróconą” parą daje błąd
# mały, ale widoczny; falka z innej rodziny daje obraz nie do przyjęcia.

bior4.4 PSNR: 141.27 dB
rbio4.4 PSNR: 40.89 dB
sym4 PSNR: 25.20 dB


### Ćwiczenie 11.5
Oblicz spójność wzajemną baz Haara i sym8 dla N = 256. Które pary atomów są do siebie najbardziej podobne?

### Rozwiązanie 11.5

In [17]:
M = 256
def baza(falka):
    uklad = pywt.coeffs_to_array(pywt.wavedec(np.zeros(M), falka, mode="periodization"))[1]
    return np.array([pywt.waverec(pywt.array_to_coeffs(e, uklad, output_format="wavedec"),
                                  falka, mode="periodization") for e in np.eye(M)])
A, B = baza("haar"), baza("sym8")
G = np.abs(A @ B.T)
print("spójność:", G.max().round(4), " para:", np.unravel_index(G.argmax(), G.shape))
G[0, 0] = 0                                    # bez wspólnej funkcji stałej
i, j = np.unravel_index(G.argmax(), G.shape)
print("następna para:", (i, j), G[i, j].round(3))
print("odsetek par o iloczynie skalarnym powyżej 0,5:", f"{100*np.mean(G > 0.5):.3f}%")

spójność: 0.8073  para: (np.int64(128), np.int64(128))
następna para: (np.int64(128), np.int64(128)) 0.807
odsetek par o iloczynie skalarnym powyżej 0,5: 0.208%


### Ćwiczenie 11.6
Zastąp w kodzie 11.7 część schodkową trzema impulsami, a bazę Haara bazą impulsów. Czy pogoń za dopasowaniem rozdziela teraz składniki poprawnie?

### Rozwiązanie 11.6

In [18]:
from scipy.fft import dct, idct
jedn = np.eye(N)
impulsy = np.zeros(N); impulsy[[100, 480, 801]] = [3.0, -2.0, 2.5]
oscylacje = 30*idct(jedn[75], norm="ortho") + 20*idct(jedn[181], norm="ortho")
s = impulsy + oscylacje
reszta, a, b, kroki = s.copy(), np.zeros(N), np.zeros(N), 0
while np.linalg.norm(reszta) > 1e-6*np.linalg.norm(s) and kroki < 100:
    ck = dct(reszta, norm="ortho")
    i, j = np.argmax(np.abs(reszta)), np.argmax(np.abs(ck))
    if abs(reszta[i]) >= abs(ck[j]):
        a[i] += reszta[i]; reszta = reszta.copy(); reszta[i] = 0
    else:
        b[j] += ck[j]; atom = np.zeros(N); atom[j] = ck[j]
        reszta = reszta - idct(atom, norm="ortho")
    kroki += 1
print("kroków:", kroki, " impulsów:", int(np.sum(np.abs(a) > 1e-6)),
      " kosinusoid:", int(np.sum(np.abs(b) > 1e-6)))
print("błąd impulsów:", round(float(np.linalg.norm(a - impulsy)/np.linalg.norm(impulsy)), 6),
      " błąd oscylacji:", round(float(np.linalg.norm(idct(b, norm="ortho") - oscylacje)
                                      / np.linalg.norm(oscylacje)), 6))

kroków: 13  impulsów: 3  kosinusoid: 2
błąd impulsów: 8e-06  błąd oscylacji: 0.0


### Ćwiczenie 11.7
Zbadaj, jak wynik uśredniania po przesunięciach w kodzie 11.9 zależy od liczby przesunięć (1, 2, 4, 8, 16, 32).

### Rozwiązanie 11.7

In [19]:
rng = np.random.default_rng(4)
s = sygnaly_testowe()["gładki ze skokami"]
szumy = [rng.normal(size=N) for _ in range(10)]
for ile in (1, 2, 4, 8, 16, 32):
    w = np.mean([snr(s, np.mean([np.roll(odszum(np.roll(s + e, k), "sym8"), -k)
                                 for k in range(ile)], axis=0)) for e in szumy])
    print(f"przesunięć: {ile:2d}  SNR {w:.2f} dB")
# Większość zysku daje już kilka przesunięć; powyżej 16 poprawa jest znikoma.

przesunięć:  1  SNR 27.67 dB
przesunięć:  2  SNR 27.85 dB
przesunięć:  4  SNR 28.26 dB
przesunięć:  8  SNR 28.56 dB
przesunięć: 16  SNR 28.76 dB
przesunięć: 32  SNR 28.94 dB


### Ćwiczenie 11.8
Wykonaj porównanie z kodu 11.1 na własnych danych: wybierz czysty zapis, dodaj szum o znanym odchyleniu i wyznacz najlepszą falkę.

### Rozwiązanie 11.8

In [20]:
# Przykład na zapisie EKG dołączonym do PyWavelets (traktowanym jako czysty).
ekg = pywt.data.ecg().astype(float)
ekg = (ekg - ekg.mean())/ekg.std()*7
rng = np.random.default_rng(0)
wyniki = {f: np.mean([snr(ekg, odszum(ekg + rng.normal(size=ekg.size), f))
                      for proba in range(20)]) for f in falki}
for f, w in sorted(wyniki.items(), key=lambda p: -p[1]):
    print(f"{f:8s} {w:.2f} dB")

coif3    20.78 dB
bior4.4  20.60 dB
db8      20.41 dB
sym8     20.40 dB
sym4     20.38 dB
db4      19.84 dB
db2      19.64 dB
haar     18.34 dB
